# Weak authentication patterns; adversary dictionary construction  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 1.2.b: Password weakness reporter

**Skill:** Analyze Risk · **Maps to:** 1.2.B · **Language:** Python 3 (pytest)

## Problem
Build the password-weakness checker an authentication backend would use to give users actionable
feedback when they choose a new password. Given a password string, return a report dict
identifying whether the password is weak and which specific rules it violates.

## Function contract
Implement `solve(password: str) -> dict` in the code editor (see the Reference section below).

## Input
A single string, the candidate password. May be empty.

## Output
A dict with exactly two keys:
- `"weak"` (`bool`), `True` if the password fails ANY rule; `False` only if it passes all rules.
- `"reasons"` (`list[str]`), sorted list of rule-labels for each rule the password fails.
  Empty list if the password passes all rules.

Possible reason labels (exact strings, lowercase):
- `"too_short"`, length < 12 characters
- `"in_dictionary"`, password (lowercased) is in the built-in common wordlist
- `"only_lowercase"`, contains no uppercase letters, digits, or non-alphabetic characters
- `"sequential_or_repeated"`, contains a sequential run of 3+ characters (e.g. `abc`, `123`,
  `xyz`) OR a repeated run of 3+ identical characters (e.g. `aaa`, `111`)

## Examples
```
solve("password")
-> {"weak": True, "reasons": ["in_dictionary", "only_lowercase", "too_short"]}

solve("P@ssword!2024")
-> {"weak": True, "reasons": ["in_dictionary"]}

solve("Tr0ub4dor&3abcde")
-> {"weak": True, "reasons": ["sequential_or_repeated"]}

solve("mXk9#vLq2@nP")
-> {"weak": False, "reasons": []}
```

## Constraints
- Deterministic, offline, standard library only.
- The common wordlist is defined in the Reference section below and must be embedded in your solution.
- Reasons list is always sorted ascending.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(password: str) -> dict
```

### Input (`password`)
A single `str`, the candidate password. May be empty or contain any Unicode.

### Output (`result`)
A `dict` with exactly these keys:
- `"weak"` (`bool`), `True` if any rule below is triggered; `False` if all rules pass.
- `"reasons"` (`list[str]`), sorted list of triggered rule labels (see below). Empty if no rules triggered.

### Rules (all must be checked; collect all failures)

#### Rule 1: `"too_short"`
Triggered when `len(password) < 12`.

#### Rule 2: `"in_dictionary"`
Triggered when `password.lower()` is a member of the COMMON_WORDS set below.
```
COMMON_WORDS = {
    "password", "password1", "password123", "123456", "123456789", "12345678",
    "qwerty", "qwerty123", "letmein", "iloveyou", "admin", "welcome",
    "monkey", "dragon", "master", "abc123", "sunshine", "princess",
    "shadow", "superman", "football", "baseball", "login", "passw0rd",
    "pa$$word", "access", "mustang", "1234567890", "starwars", "hello",
}
```

#### Rule 3: `"only_lowercase"`
Triggered when the password contains ONLY lowercase ASCII letters (`a`-`z`), i.e.,
`password.isalpha() and password == password.lower()`.
Note: an empty string does NOT trigger this rule (it has no lowercase letters either).

#### Rule 4: `"sequential_or_repeated"`
Triggered when the password contains:
- A run of 3 or more consecutive ASCII characters in ascending order
  (e.g. `abc`, `bcd`, `xyz`, `123`, `234`, `789`), OR
- A run of 3 or more identical characters (e.g. `aaa`, `111`, `   `)

Detection algorithm:
For each window of 3 consecutive characters at position i:
- Repeated: `c[i] == c[i+1] == c[i+2]`
- Sequential ascending: `ord(c[i+1]) - ord(c[i]) == 1` AND `ord(c[i+2]) - ord(c[i+1]) == 1`

If any window matches either condition, the rule is triggered.

### Output contract
- `"reasons"` is always sorted ascending (alphabetical order of the label strings).
- `"weak"` equals `len(reasons) > 0`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- The COMMON_WORDS set above is the exact set used by tests.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 1.2.b. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""

COMMON_WORDS = {
    "password", "password1", "password123", "123456", "123456789", "12345678",
    "qwerty", "qwerty123", "letmein", "iloveyou", "admin", "welcome",
    "monkey", "dragon", "master", "abc123", "sunshine", "princess",
    "shadow", "superman", "football", "baseball", "login", "passw0rd",
    "pa$$word", "access", "mustang", "1234567890", "starwars", "hello",
}


def solve(password: str) -> dict:
    """Return a weakness report dict for the given password.

    Returns:
        {"weak": bool, "reasons": list[str]} where reasons is a sorted list of
        triggered rule labels from {"too_short","in_dictionary","only_lowercase",
        "sequential_or_repeated"}.
    """
    reasons = []

    # Rule 1: too_short
    if len(password) < 12:
        reasons.append("too_short")

    # Rule 2: in_dictionary
    if password.lower() in COMMON_WORDS:
        reasons.append("in_dictionary")

    # Rule 3: only_lowercase
    # Triggered iff every character is a lowercase ASCII letter (non-empty)
    if password and password.isalpha() and password == password.lower():
        reasons.append("only_lowercase")

    # Rule 4: sequential_or_repeated
    # Scan every window of 3 consecutive characters
    triggered = False
    for i in range(len(password) - 2):
        a, b, c = password[i], password[i + 1], password[i + 2]
        oa, ob, oc = ord(a), ord(b), ord(c)
        if a == b == c:
            triggered = True
            break
        if ob - oa == 1 and oc - ob == 1:
            triggered = True
            break
    if triggered:
        reasons.append("sequential_or_repeated")

    reasons.sort()
    return {"weak": len(reasons) > 0, "reasons": reasons}


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------

def test_dict_only_lc_short():
    """'password' is short, in dictionary, and only lowercase."""
    result = impl.solve("password")
    assert result["weak"] is True
    assert result["reasons"] == ["in_dictionary", "only_lowercase", "too_short"]


def test_sequential_only():
    """'Tr0ub4dor&3abcde' is long, mixed, not in dictionary, but has 'abc'."""
    result = impl.solve("Tr0ub4dor&3abcde")
    assert result["weak"] is True
    assert result["reasons"] == ["sequential_or_repeated"]


def test_strong():
    """'mXk9#vLq2@nP' passes all rules."""
    result = impl.solve("mXk9#vLq2@nP")
    assert result["weak"] is False
    assert result["reasons"] == []


def test_dictionary_with_mangling():
    """'P@ssword!2024' is long and mixed but base 'passw0rd' is in dictionary."""
    # Note: 'passw0rd' IS in COMMON_WORDS; 'P@ssword!2024'.lower() = 'p@ssword!2024' is NOT.
    # However 'pa$$word' IS: test a direct dictionary hit with symbols.
    result = impl.solve("pa$$word")
    assert result["weak"] is True
    assert "in_dictionary" in result["reasons"]
    assert "too_short" in result["reasons"]


# ---- hidden / edge cases -------------------------------------------------

def test_empty_string():
    """Empty string is too_short only (not only_lowercase, not in dictionary)."""
    result = impl.solve("")
    assert result["weak"] is True
    assert result["reasons"] == ["too_short"]


def test_repeated_chars():
    """'HelloWorld111' triggers sequential_or_repeated due to '111'."""
    result = impl.solve("HelloWorld111")
    assert result["weak"] is True
    assert result["reasons"] == ["sequential_or_repeated"]


def test_only_lowercase_flag():
    """All-lowercase short word triggers both too_short and only_lowercase."""
    result = impl.solve("sunshine")
    assert "only_lowercase" in result["reasons"]
    assert "too_short" in result["reasons"]
    assert "in_dictionary" in result["reasons"]


def test_reasons_sorted():
    """Reasons list is always sorted ascending regardless of check order."""
    result = impl.solve("qwerty")
    assert result["reasons"] == sorted(result["reasons"])


def test_sequential_digits():
    """'Abcdef123456!' is long and mixed but contains '123' sequential run."""
    result = impl.solve("Abcdef123456!")
    assert result["weak"] is True
    assert "sequential_or_repeated" in result["reasons"]


def test_exactly_12_chars_passes_length():
    """A 12-character password that is otherwise clean should not trigger too_short."""
    # 'zQm7!kLp3$Rw' -- 12 chars, mixed, not in dictionary, no sequences
    result = impl.solve("zQm7!kLp3$Rw")
    assert "too_short" not in result["reasons"]


def test_weak_flag_matches_reasons():
    """'weak' must be True iff reasons is non-empty."""
    for pw in ["mXk9#vLq2@nP", "password", "HelloWorld111", ""]:
        result = impl.solve(pw)
        assert result["weak"] == (len(result["reasons"]) > 0)


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
